# LLM Project 2 — AI Workspace Agent Suite
## Google Workspace MCP · LangGraph ReAct · Gmail + Calendar APIs — Demo Notebook

**LLM**: OpenAI `gpt-4o` | **Framework**: LangGraph StateGraph | **Workspace**: Google Workspace MCP (stdio)

> This notebook is the authoritative demo notebook for Project 2. All code is self-contained.
> Interactive mode cells are commented out — they block the Jupyter kernel.

---

## PDF Section Map

| PDF Section | Title | Notebook Treatment |
| --- | --- | --- |
| 1 | Project Overview | Markdown only |
| 2 | Projects in the Suite | Markdown only |
| 3 | Technology Stack | Markdown only |
| 4 | System Architecture | Markdown + graph compiled in Section 6 |
| 5 | Tool Inventory | Markdown + tools loaded at runtime |
| 6 | Component Descriptions | Code — all components defined |
| 7 | Project A: Refund Email Agent | Code — AUTO mode demo |
| 8 | Project B: Calendar Agent | Code — DEMO mode (3 testing-spec prompts) |
| 9 | Setup Instructions | Code — prerequisites check |
| 10 | Security Design | Markdown only |
| 11 | Key Concepts Reference | Markdown only |

---

## Demo Run Order (all cells can be run from within this notebook)

| Step | Cell | Action |
| --- | --- | --- |
| 1 | Prerequisites check | Verifies `workspace-cli`, `uvx`, env vars |
| 2 | Verify Google OAuth | Runs `workspace-cli call list_calendars` — opens browser once if needed |
| 3 | Seed Gmail inbox | Sends 8 test emails via SMTP (fill credentials in `.env`) |
| 4 | Refund Agent AUTO | Processes inbox autonomously, prints summary report |
| 5 | Calendar Agent DEMO | Runs 3 testing-spec queries automatically |

> **Only true prerequisite**: `workspace-cli` must be installed and `.env` must have `OPENAI_API_KEY` + Google OAuth + SMTP credentials.

## Cannot be Demonstrated Automatically

| Feature | Reason |
| --- | --- |
| Interactive chat mode | Uses `input()` — blocks Jupyter kernel |
| First OAuth browser flow | Requires browser interaction (one-time only) |
| Email sending to real recipients | Requires Gmail scope + real inbox |

---
## 1 — Project Overview *(Informational — no code required)*

A suite of two autonomous AI agents connected to a **live Google Workspace account**:

| Agent | File | Task |
| --- | --- | --- |
| **Project A — Refund Email Agent** | `refund_agent.py` / Section 7 | Monitors Gmail inbox, classifies refund/return emails, sends threaded replies — fully autonomous |
| **Project B — Calendar Agent** | `calendar_agent.py` / Section 8 | Answers natural language questions about Google Calendar, creates/modifies events, checks free slots, sends RSVPs |

Both agents share the same core architecture: **ReAct pattern** via LangGraph `StateGraph`, powered by **OpenAI gpt-4o-mini**, connected to Google Workspace through the open-source `google_workspace_mcp` server.

The Calendar Agent adds a second tool layer: **workspace-cli bash tools** — direct subprocess calls for fast, lightweight calendar reads.

---
## 2 — Projects in the Suite *(Informational — no code required)*

### Project A — Refund Email Agent

Automated **6-step workflow**:
```
SEARCH inbox → READ each email → CLASSIFY intent →
DRAFT reply from template → SEND threaded reply → REPORT summary
```

| Class | Description | Agent Action |
| --- | --- | --- |
| `REFUND_REQUEST` | Customer wants money back | Send refund approval reply (3–5 day processing) |
| `RETURN_REQUEST` | Customer wants to return product | Send return instructions with prepaid label steps |
| `COMPLAINT` | General dissatisfaction | Send empathetic acknowledgement, 24hr follow-up promise |
| `OTHER` | Unrelated content | Skip — no reply sent |

### Project B — Calendar Agent

**Dual tool strategy**:
```
Simple read  →  workspace-cli bash tool  (fast subprocess, minimal overhead)
Create/Edit  →  Calendar MCP tool        (full CRUD, rich JSON response)
```

The model selects the appropriate tool surface based on the task type.

---
## 3 — Technology Stack *(Informational — no code required)*

| Component | Technology |
| --- | --- |
| Language | Python 3.11+ |
| LLM | OpenAI `gpt-4o-mini` (OpenAI API) |
| Agent Framework | LangGraph `StateGraph` + `ToolNode` |
| Tool Protocol | Model Context Protocol (MCP) — open standard, Linux Foundation |
| MCP Server | `google_workspace_mcp` — `github.com/taylorwilsdon/google_workspace_mcp` |
| CLI Tool | `workspace-cli` — built into same repo, installed via `uv tool install .` |
| Gmail Access | Google Gmail API via OAuth 2.0 |
| Calendar Access | Google Calendar API via OAuth 2.0 |
| Auth | Google Cloud OAuth 2.0 Desktop App flow |
| Transport | `stdio` (local subprocess) — MCP JSON-RPC over stdin/stdout |

**Python packages**: `langgraph`, `langchain-openai`, `langchain-mcp-adapters`, `langchain-core`, `nest_asyncio`

---
## 4 — System Architecture — ReAct Graph *(Code: graph compiled in Sections 7 & 8)*

Both agents share the same **three-node LangGraph topology**:

```text
User Input (HumanMessage)
        │
        ▼
┌───────────────────────────────────────┐
│            agent_node                 │
│  Prepends SYSTEM_PROMPT to history    │
│  Calls nemotron-super-fp8 with full message state │
│  Returns text OR tool calls           │
└──────────────┬────────────────────────┘
               │
       ┌───────▼──────────┐
       │  should_continue │   (conditional edge)
       └──┬───────────────┘
          │                     │
    has tool_calls          no tool_calls
          │                     │
          ▼                     ▼
  ┌──────────────┐           ┌─────┐
  │  tool_node   │           │ END │
  │  Executes    │           └─────┘
  │  MCP or CLI  │
  │  tool call   │
  └──────┬───────┘
         │   ToolMessage appended to state
         └───────────────────────────────► agent_node (loop)
```

**Key principle**: The loop continues until the model produces a response with no tool calls. A single user question may trigger 5–15 tool calls internally before the agent produces its final answer.

---
## 9 — Setup Instructions

### Step 1 — Clone and install workspace-mcp + CLI
```bash
git clone https://github.com/taylorwilsdon/google_workspace_mcp
cd google_workspace_mcp
uv tool install .          # installs workspace-cli globally (requires uv)
pip install workspace-mcp  # installs the MCP server Python package
```

### Step 2 — Install Python dependencies (already in .venv)
```bash
pip install langgraph langchain-openai langchain-mcp-adapters nest_asyncio python-dotenv
```

### Steps 3–5 — Google Cloud OAuth (see Section 5 markdown)

### Step 6 — Verify CLI before running agents
```bash
workspace-cli list                 # list all available tools
workspace-cli call list_calendars  # verify OAuth works
```

In [ ]:
# Install Python dependencies (run once if not already installed)
# langchain-openai is a LangChain adapter that connects to the vLLM REST API on the DGX Spark
import subprocess, sys
pkgs = ['langgraph', 'langchain-openai', 'langchain-mcp-adapters', 'nest_asyncio', 'python-dotenv']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=True)
print('Python packages ready.')
print()
print('NOTE: workspace-cli must be installed separately:')
print('  git clone https://github.com/taylorwilsdon/google_workspace_mcp')
print('  cd google_workspace_mcp && uv tool install .')
print('  pip install workspace-mcp')

---
## Prerequisites Check

Run the cell below to verify all required tools and credentials are present before executing the agents.

In [ ]:
import subprocess, os, shutil

print('=== Pre-Demo Prerequisites Check ===')
print()

# 1. workspace-cli
cli_ok = shutil.which('workspace-cli') is not None
print(f'workspace-cli installed : {"OK" if cli_ok else "MISSING — run: cd google_workspace_mcp && uv tool install ."}')

# 2. uvx
uvx_ok = shutil.which('uvx') is not None
print(f'uvx installed           : {"OK" if uvx_ok else "MISSING — install uv: curl -LsSf https://astral.sh/uv/install.sh | sh"}')

# 3. Required env vars
from dotenv import load_dotenv
load_dotenv()

openai_key = os.getenv('OPENAI_API_KEY', '')
goog_id = os.getenv('GOOGLE_OAUTH_CLIENT_ID', '')
goog_sec = os.getenv('GOOGLE_OAUTH_CLIENT_SECRET', '')

print(f'OPENAI_API_KEY          : {"SET" if openai_key and openai_key != "your_openai_api_key_here" else "MISSING — set in .env"}')
print(f'GOOGLE_OAUTH_CLIENT_ID  : {"SET" if goog_id and "your" not in goog_id else "MISSING — set in .env"}')
print(f'GOOGLE_OAUTH_CLIENT_SECRET: {"SET" if goog_sec and "your" not in goog_sec else "MISSING — set in .env"}')

# 4. Quick CLI test
print()
if cli_ok:
    try:
        result = subprocess.run(['workspace-cli', 'list'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            tool_count = len([l for l in result.stdout.split('\n') if l.strip()])
            print(f'workspace-cli tools available: {tool_count} tools found')
        else:
            print(f'workspace-cli list failed: {result.stderr[:100]}')
    except Exception as e:
        print(f'workspace-cli test error: {e}')

print()
print('If all checks pass, proceed to run the agents below.')

---
### Step 1 — Verify Google OAuth

Run the cell below. If not yet authenticated it will **open a browser window** (one-time). Once done, it prints your calendars and confirms the connection is working.

In [ ]:
import subprocess, json, shutil

if not shutil.which('workspace-cli'):
    print('workspace-cli not found — install it first (see prerequisites above)')
else:
    print('Running: workspace-cli call list_calendars')
    print('(If not yet authenticated, a browser window will open — complete the OAuth flow there)')
    print()
    result = subprocess.run(
        ['workspace-cli', 'call', 'list_calendars'],
        capture_output=True, text=True,
        timeout=120  # 2 minutes for browser OAuth flow
    )
    if result.returncode == 0:
        try:
            data = json.loads(result.stdout)
            calendars = data if isinstance(data, list) else data.get('calendars', data)
            print(f'OAuth OK — {len(calendars) if isinstance(calendars, list) else "?"} calendar(s) found:')
            if isinstance(calendars, list):
                for cal in calendars:
                    name = cal.get('summary', cal.get('id', str(cal)))
                    print(f'  - {name}')
        except Exception:
            print('OAuth OK — raw output:')
            print(result.stdout[:500])
    else:
        print(f'Error (exit {result.returncode}):')
        print(result.stderr[:300])

---
## Google Cloud OAuth 2.0 Setup — Step-by-Step

> **What this gives you**: A `Client ID` and `Client Secret` that let the agents log in to your Google account and access Gmail + Calendar on your behalf.

---

### Part 1 — Create a Google Cloud Project

1. Open [console.cloud.google.com](https://console.cloud.google.com) and sign in with the Google account you want to use.
2. At the very top of the page, click the **project selector dropdown** (it shows "Select a project" or an existing project name).
3. In the popup, click **"New Project"** (top-right corner).
4. Enter a **Project name** — e.g. `AI Workspace Agent` — then click **"Create"**.
5. Wait ~10 seconds. A notification appears bottom-right. Click **"Select Project"** in it (or use the top dropdown again to switch to your new project).

---

### Part 2 — Enable Gmail and Calendar APIs

6. In the left sidebar, click **"APIs & Services"** → **"Library"**.
7. In the search box, type **`Gmail API`** → click the result → click the blue **"Enable"** button.
8. Click the **back arrow** in your browser.
9. Search for **`Google Calendar API`** → click the result → click **"Enable"**.

---

### Part 3 — Configure the OAuth Consent Screen

> This is the screen users see when they grant your app access. Even for personal use you must set it up once.

10. In the left sidebar, click **"APIs & Services"** → **"OAuth consent screen"**.
11. Under "User Type", select **"External"** → click **"Create"**.
12. Fill in the required fields:
    - **App name**: `AI Workspace Agent` (any name)
    - **User support email**: select your email from the dropdown
    - **Developer contact information** (bottom): type your email
    - Leave everything else blank
13. Click **"Save and Continue"**.
14. On the **Scopes** page, click **"Add or Remove Scopes"**.
15. In the filter box, search for and tick each of these scopes one at a time:

    | Scope to find | What to search |
    | --- | --- |
    | `https://mail.google.com/` | `mail.google` |
    | `https://www.googleapis.com/auth/gmail.send` | `gmail.send` |
    | `https://www.googleapis.com/auth/gmail.modify` | `gmail.modify` |
    | `https://www.googleapis.com/auth/calendar` | `calendar` (full) |
    | `https://www.googleapis.com/auth/calendar.events` | `calendar.events` |

16. Click **"Update"** → then **"Save and Continue"**.
17. On the **Test users** page, click **"+ Add Users"** → type your Google account email → click **"Add"**.
18. Click **"Save and Continue"** → then **"Back to Dashboard"**.

---

### Part 4 — Create OAuth 2.0 Credentials

19. In the left sidebar, click **"APIs & Services"** → **"Credentials"**.
20. Click **"+ Create Credentials"** (top of page) → select **"OAuth client ID"**.
21. Under **"Application type"**, select **"Desktop app"**.
22. Give it a name: `AI Workspace Agent Desktop` (any name).
23. Click **"Create"**.
24. A popup appears showing your **Client ID** and **Client Secret**.
    - Copy the **Client ID** (long string ending in `.apps.googleusercontent.com`)
    - Copy the **Client Secret** (shorter string)
25. Click **"OK"** to close the popup.

> You can always come back to **Credentials** page to view or re-copy these values.

---

### Part 5 — Save to .env

Add the values you just copied to your `.env` file in this project:

```bash
GOOGLE_OAUTH_CLIENT_ID=xxxx.apps.googleusercontent.com
GOOGLE_OAUTH_CLIENT_SECRET=GOCSPX-xxxx
OAUTHLIB_INSECURE_TRANSPORT=1
```

Or paste them directly into the **Credentials cell** below (uncomment the lines).

---

### Part 6 — What Happens on First Run

When you run either agent for the first time, a **browser window opens automatically** asking you to sign in to Google and approve access. Steps:

1. Sign in with the same Google account you added as a test user (Step 17 above).
2. You will see a warning: **"Google hasn't verified this app"** — click **"Continue"** (this is expected for apps in testing mode).
3. Check all the permission boxes → click **"Continue"**.
4. The browser shows `"The authentication flow has completed"` — you can close it.
5. Tokens are saved to `~/.workspace-mcp/` (encrypted). You will **not** be asked again unless you revoke access.

---

### Troubleshooting

| Problem | Fix |
| --- | --- |
| "Access blocked: app not verified" | On the warning screen, click **"Advanced"** → **"Go to AI Workspace Agent (unsafe)"** |
| "redirect_uri_mismatch" | You chose "Web application" instead of "Desktop app" in Step 21 — recreate the credential |
| Browser doesn't open | Run `workspace-cli call list_calendars` in a terminal to trigger the OAuth flow manually |
| "invalid_client" error | Double-check that Client ID and Secret are copied exactly, no extra spaces |

---
## Credentials — Environment Variables

Set your credentials here before running either agent.

> **Security**: These are injected as environment variables. Never commit actual secrets to git.
> The `.env` file (gitignored) is the recommended approach for persistent storage.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # load from .env if present

# ── OpenAI endpoint ──────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = os.getenv('OPENAI_MODEL', 'gpt-4o')

# ── Google OAuth — fill in .env or uncomment below ───────────────────────────
# os.environ['GOOGLE_OAUTH_CLIENT_ID']     = '<your-client-id>'
# os.environ['GOOGLE_OAUTH_CLIENT_SECRET'] = '<your-client-secret>'
# os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'   # local dev only
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED = ['GOOGLE_OAUTH_CLIENT_ID', 'GOOGLE_OAUTH_CLIENT_SECRET']
missing  = [k for k in REQUIRED if not os.environ.get(k)]

if missing:
    print('MISSING credentials (set in .env or uncomment above):')
    for k in missing:
        print(f'  {k}')
    print()
    print('Agents will NOT run until Google OAuth credentials are set.')
else:
    print('All credentials present.')
    print(f'  OPENAI_MODEL              : {OPENAI_MODEL}')
    print(f'  OPENAI_API_KEY            : {OPENAI_API_KEY[:8] if OPENAI_API_KEY else "NOT SET"}...')
    print(f'  GOOGLE_OAUTH_CLIENT_ID    : {os.environ["GOOGLE_OAUTH_CLIENT_ID"][:12]}...')
    print(f'  GOOGLE_OAUTH_CLIENT_SECRET: {os.environ["GOOGLE_OAUTH_CLIENT_SECRET"][:8]}...')


In [ ]:
import os
import json
import asyncio
import subprocess
from datetime import datetime, timezone, timedelta
from typing import Annotated, Sequence, Any
from typing_extensions import TypedDict

import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run() inside Jupyter's running event loop

from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, BaseMessage, add_messages
)
from langchain_openai import ChatOpenAI  # LangChain adapter for vLLM REST API
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

print('All imports OK.')
print(f'LangGraph + LangChain + MCP adapters loaded.')

---
## 5 — Tool Inventory

### 5.1 Gmail MCP Tools — Refund Email Agent

Loaded from the running `workspace-mcp` server via `mcp_client.get_tools()` and filtered by name.

| Tool Name | Purpose | Key Parameters |
| --- | --- | --- |
| `search_gmail_messages` | Search inbox with Gmail query operators | `query`, `max_results` |
| `get_gmail_message_content` | Read full body + metadata of one email | `message_id` |
| `get_gmail_messages_content_batch` | Fetch up to 25 emails in one call | `message_ids`, `format` |
| `send_gmail_message` | Send or reply to an email (threaded) | `to`, `subject`, `body`, `thread_id` |
| `create_gmail_draft` | Save a draft for human review | `to`, `subject`, `body` |
| `get_gmail_thread` | Read full conversation thread | `thread_id` |
| `list_gmail_labels` | List all Gmail labels and folders | *(none)* |

### 5.2 Calendar MCP Tools — Calendar Agent

| Tool Name | Purpose | Key Parameters |
| --- | --- | --- |
| `list_calendar_events` | List events in a date range | `calendarId`, `timeMin`, `timeMax` |
| `get_calendar_event` | Get one event by ID | `calendarId`, `eventId` |
| `list_calendars` | List all calendars the user has | *(none)* |
| `create_calendar_event` | Create a new calendar event | `summary`, `start`, `end`, `attendees` |
| `update_calendar_event` | Update an existing event | `calendarId`, `eventId`, `updates` |
| `delete_calendar_event` | Delete an event | `calendarId`, `eventId` |
| `suggest_meeting_time` | Find free slots across attendees | `attendees`, `duration` |
| `respond_to_calendar_event` | RSVP accept / decline / tentative | `calendarId`, `eventId`, `response` |

### 5.3 workspace-cli Bash Tools — Calendar Agent Only

Python `@tool`-decorated functions that call `workspace-cli` as a local subprocess. Used for fast, low-overhead calendar reads.

| Tool Name | CLI Command | Use Case |
| --- | --- | --- |
| `cli_today_events` | `workspace-cli call list_calendar_events` (today's ISO range) | "What's on today?" |
| `cli_list_events` | `workspace-cli call list_calendar_events` (custom range) | "Show me this week" |
| `cli_list_calendars` | `workspace-cli call list_calendars` | "What calendars do I have?" |
| `cli_get_event` | `workspace-cli call get_calendar_event --eventId <id>` | "Get details for that meeting" |
| `cli_tool_list` | `workspace-cli list` | Debug / tool discovery |

---
## 6 — Component Descriptions

### 6.1 AgentState — Shared Memory (Both Agents)

A Python `TypedDict` that holds the **full conversation history** for the running agent session.
The `add_messages` reducer appends new messages rather than replacing the list — every tool result, AI response, and user message accumulates across the entire ReAct loop.

This is the "working memory" that allows multi-step reasoning to function.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

print('AgentState defined (shared by both agents).')
print('  messages: Sequence[BaseMessage] with add_messages reducer')

---
## 7 — Project A: Refund Email Agent

An autonomous customer service agent that monitors a Gmail inbox for refund and return request emails, classifies them, composes professional replies using pre-defined templates, and sends threaded replies — **without any human involvement beyond the initial run command**.

### 6.2 WORKSPACE_MCP_CONFIG — Gmail Agent

Tells `MultiServerMCPClient` how to spawn and communicate with the `workspace-mcp` subprocess.

- `--single-user` — OAuth 2.0 for one Google account
- `--tool-tier core` — loads only essential tools (~20 vs 100+)
- `--permissions gmail:send` — restricts scope to Gmail only
- `transport: stdio` — local pipe, no network, data never leaves the machine

In [ ]:
GMAIL_MCP_CONFIG = {
    'workspace': {
        'command': 'uvx',
        'args': [
            'workspace-mcp',
            '--single-user',
            '--tool-tier', 'core',
            '--permissions', 'gmail',
        ],
        'transport': 'stdio',
        'env': {
            'GOOGLE_OAUTH_CLIENT_ID':     os.environ.get('GOOGLE_OAUTH_CLIENT_ID', ''),
            'GOOGLE_OAUTH_CLIENT_SECRET': os.environ.get('GOOGLE_OAUTH_CLIENT_SECRET', ''),
        },
    }
}

GMAIL_TOOL_NAMES = [
    'search_gmail_messages',
    'get_gmail_message_content',
    'get_gmail_messages_content_batch',
    'send_gmail_message',
    'create_gmail_draft',
    'get_gmail_thread',
    'list_gmail_labels',
]

print('Gmail MCP config ready.')
print(f'  Permissions : gmail:send')
print(f'  Tool tier   : core')
print(f'  Transport   : stdio')
print(f'  Tools expected: {GMAIL_TOOL_NAMES}')

### 6.3 SYSTEM_PROMPT — Refund Email Agent

Defines the agent's **6-step workflow**, three **reply templates** (REFUND / RETURN / COMPLAINT), and hard rules:
- Always thread replies using `thread_id`
- Never reply to `OTHER` emails
- Prefer `create_gmail_draft` over direct sends when uncertain

In [ ]:
GMAIL_SYSTEM_PROMPT = SystemMessage(content="""
You are an autonomous customer service agent processing Gmail inbox for refund and return request emails.

## Your Workflow (6 Steps)
1. SEARCH: Use search_gmail_messages with query "refund OR return OR complaint is:unread"
2. READ: Use get_gmail_message_content or get_gmail_messages_content_batch for each email
3. CLASSIFY: Determine type: REFUND_REQUEST, RETURN_REQUEST, COMPLAINT, or OTHER
4. DRAFT: Compose a reply using the appropriate template below
5. SEND: Use send_gmail_message with thread_id from the original email (for threaded reply)
6. REPORT: After all emails, provide a summary of all actions taken

## Email Templates

### REFUND_REQUEST
Dear [Customer Name],
Thank you for contacting us about your refund request. We have received your request and are happy to help.
Your refund has been approved and will be processed within 3-5 business days to your original payment method.
If you have any questions, please don't hesitate to reach out.
Best regards, Customer Service Team

### RETURN_REQUEST
Dear [Customer Name],
Thank you for reaching out about your return. We're sorry the product did not meet your expectations.
To process your return: (1) Pack the item securely. (2) Use the prepaid shipping label we will provide.
(3) Drop off at any authorized shipping location. Your refund will be processed within 5-7 business days of receipt.
Best regards, Customer Service Team

### COMPLAINT
Dear [Customer Name],
Thank you for reaching out. We sincerely apologize for the inconvenience you have experienced.
Your feedback is extremely valuable. We have escalated your concern to our quality team and will follow up
with a resolution within 24 hours. We appreciate your patience.
Best regards, Customer Service Team

## Hard Rules
- ALWAYS use thread_id when calling send_gmail_message to ensure threaded replies
- NEVER reply to OTHER emails (unrelated content)
- Prefer create_gmail_draft over send_gmail_message when classification is uncertain
- Use the customer's actual name from the email if available in the template
- After processing all emails, provide a complete summary: total processed, breakdown by class, actions taken
""")

print('Gmail system prompt defined.')
print(f'  Length: {len(GMAIL_SYSTEM_PROMPT.content)} chars')
print('  Templates: REFUND_REQUEST, RETURN_REQUEST, COMPLAINT')
print('  Hard rules: thread_id required, skip OTHER, prefer drafts when uncertain')

### 6.6 build_agent() — Agent Factory (Refund Email Agent)

Assembles the complete LangGraph agent in five steps:
1. Calls `mcp_client.get_tools()` to fetch the live tool list from the running MCP server
2. Filters tools by name to the Gmail-relevant set
3. Creates `ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY)` — points LangChain to the vLLM server on the DGX Spark — and binds all tools via `.bind_tools()`
4. Defines `agent_node`, `tool_node`, `should_continue` edges
5. Calls `.compile()` to produce the runnable graph

In [ ]:
async def build_gmail_agent(mcp_client: MultiServerMCPClient):
    """Build the Refund Email Agent LangGraph from the live MCP server tools."""
    all_tools = await mcp_client.get_tools()
    gmail_tools = [t for t in all_tools if t.name in GMAIL_TOOL_NAMES]

    print(f'MCP tools loaded: {len(all_tools)} total, {len(gmail_tools)} Gmail tools selected.')
    for t in gmail_tools:
        print(f'  - {t.name}')

    llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    llm_with_tools = llm.bind_tools(gmail_tools)

    def agent_node(state: AgentState) -> AgentState:
        """LLM reasoning node: prepend system prompt, call gpt-4o."""
        response = llm_with_tools.invoke([GMAIL_SYSTEM_PROMPT] + list(state['messages']))
        return {'messages': [response]}

    def should_continue(state: AgentState) -> str:
        """Route to tools if the model emitted tool calls, else END."""
        last = state['messages'][-1]
        if hasattr(last, 'tool_calls') and last.tool_calls:
            return 'tools'
        return END

    tool_node = ToolNode(gmail_tools)

    workflow = StateGraph(AgentState)
    workflow.add_node('agent', agent_node)
    workflow.add_node('tools', tool_node)
    workflow.set_entry_point('agent')
    workflow.add_conditional_edges('agent', should_continue)
    workflow.add_edge('tools', 'agent')

    return workflow.compile()


print('build_gmail_agent() defined.')
print('Graph: START -> agent_node <-> tool_node -> END (ReAct loop)')

### 6.10 run_auto_refund_processing() — Auto Mode

Fires one fixed `HumanMessage` instructing the agent to execute the complete 6-step workflow end-to-end.
Calls `agent.ainvoke()` and waits for the full ReAct loop to finish (typically 10–20 tool calls).
No further user input required — **fully autonomous**.

In [ ]:
async def run_auto_refund_processing(agent) -> None:
    """Fire one instruction and let the agent process all refund/return emails autonomously."""
    instruction = (
        'Process all refund and return emails in the Gmail inbox. '
        'Execute the full 6-step workflow: search, read, classify, draft reply, send, report. '
        'Provide a summary at the end showing total emails found, classification breakdown, and actions taken.'
    )

    print('Auto-processing refund and return emails...')
    print('=' * 70)

    result = await agent.ainvoke({'messages': [HumanMessage(content=instruction)]})

    final = next(
        (m.content for m in reversed(result['messages'])
         if isinstance(m, AIMessage) and not getattr(m, 'tool_calls', None)),
        'No summary response generated.'
    )

    tool_calls_made = sum(
        len(getattr(m, 'tool_calls', []))
        for m in result['messages']
        if isinstance(m, AIMessage)
    )

    print(f'Total tool calls made: {tool_calls_made}')
    print()
    print('Agent Summary:')
    print('-' * 60)
    print(final)


print('run_auto_refund_processing() defined.')

### 6.12 run_interactive_chat() — Interactive Mode (Refund Agent)

Runs a `while True` CLI input loop. Maintains a `history` list that grows with each turn, enabling multi-turn conversation. Passes the full history into every `agent.ainvoke()` call. Exits on `"exit"` or `"quit"`.

In [ ]:
async def run_interactive_gmail_chat(agent) -> None:
    """Interactive multi-turn chat with the Refund Email Agent."""
    history: list[BaseMessage] = []
    print('Refund Email Agent — Interactive Mode')
    print('Commands: type your query | "exit" to quit | "auto" to run auto-processing')
    print('=' * 60)

    while True:
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            break

        if not user_input:
            continue
        if user_input.lower() in ('exit', 'quit'):
            print('Goodbye!')
            break
        if user_input.lower() == 'auto':
            await run_auto_refund_processing(agent)
            continue

        history.append(HumanMessage(content=user_input))
        result = await agent.ainvoke({'messages': history})
        history = list(result['messages'])

        response = next(
            (m.content for m in reversed(history)
             if isinstance(m, AIMessage) and not getattr(m, 'tool_calls', None)),
            '(no response)'
        )
        print(f'Agent: {response}')
        print()


print('run_interactive_gmail_chat() defined.')

### 6.13 main() — Orchestrator (Refund Email Agent)

Top-level entry point. Validates environment variables, opens `MultiServerMCPClient` as an `async with` context manager (which spawns and keeps alive the `workspace-mcp` subprocess), calls `build_gmail_agent()`, then runs in auto or interactive mode.

---
### Step 2 — Seed Gmail Inbox with 8 Test Emails

The Refund Agent needs emails to process. Fill in your Gmail sender credentials below and run this cell to send 8 test emails (2 REFUND_REQUEST, 2 RETURN_REQUEST, 2 COMPLAINT, 2 OTHER).

> **Note**: `SENDER_EMAIL` is the Gmail account that sends the test emails (needs an App Password). `TARGET_EMAIL` is the inbox the Refund Agent monitors (your Google account connected via OAuth).

In [ ]:
import smtplib, time, os
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv
load_dotenv()

# ── Fill in your credentials here (or add to .env) ──────────────────────────
SENDER_EMAIL  = os.getenv('TEST_SENDER_EMAIL', 'your_test_account@gmail.com')
APP_PASSWORD  = os.getenv('TEST_SENDER_APP_PASSWORD', 'your_gmail_app_password')
TARGET_EMAIL  = os.getenv('TEST_TARGET_EMAIL', 'your_agent_mailbox@gmail.com')
# ─────────────────────────────────────────────────────────────────────────────

TEST_EMAILS = [
    {'subject': 'Refund Request for Order #1001',
     'body': 'I would like to request a refund for Order #1001. The product does not meet my expectations.',
     'category': 'REFUND_REQUEST'},
    {'subject': 'Refund Request for Order #1002',
     'body': 'The item arrived damaged. Please process a refund.',
     'category': 'REFUND_REQUEST'},
    {'subject': 'Return Request for Wireless Mouse',
     'body': 'I would like to return my wireless mouse. Please send return instructions.',
     'category': 'RETURN_REQUEST'},
    {'subject': 'Return Request for Keyboard',
     'body': 'The keyboard is incompatible with my system. I would like to return it.',
     'category': 'RETURN_REQUEST'},
    {'subject': 'Very Disappointed',
     'body': 'Your customer service has been extremely disappointing. I expect a response immediately.',
     'category': 'COMPLAINT'},
    {'subject': 'Poor Service Experience',
     'body': 'I have contacted support multiple times and nobody helped me.',
     'category': 'COMPLAINT'},
    {'subject': 'Special Summer Promotion',
     'body': 'Check out our newest products and discounts. Marketing Team',
     'category': 'OTHER'},
    {'subject': 'Question About Refund Policy',
     'body': 'Before purchasing, I would like to know your refund policy.',
     'category': 'OTHER'},
]

if 'your_' in SENDER_EMAIL or 'your_' in APP_PASSWORD or 'your_' in TARGET_EMAIL:
    print('Fill in SENDER_EMAIL, APP_PASSWORD, and TARGET_EMAIL above (or add to .env)')
    print('  TEST_SENDER_EMAIL=xxx@gmail.com')
    print('  TEST_SENDER_APP_PASSWORD=xxxx xxxx xxxx xxxx')
    print('  TEST_TARGET_EMAIL=agent_inbox@gmail.com')
else:
    sent, failed = 0, 0
    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, APP_PASSWORD)
        print(f'Connected to smtp.gmail.com. Sending {len(TEST_EMAILS)} test emails to {TARGET_EMAIL}...\n')

        for idx, e in enumerate(TEST_EMAILS):
            try:
                msg = MIMEMultipart()
                msg['From'], msg['To'], msg['Subject'] = SENDER_EMAIL, TARGET_EMAIL, e['subject']
                msg.attach(MIMEText(e['body'], 'plain'))
                server.sendmail(SENDER_EMAIL, TARGET_EMAIL, msg.as_string())
                print(f'[{idx+1}/{len(TEST_EMAILS)}] OK  {e["category"]:<20} {e["subject"]}')
                sent += 1
                time.sleep(0.5)
            except smtplib.SMTPException as ex:
                print(f'[{idx+1}/{len(TEST_EMAILS)}] FAIL {e["subject"]}: {ex}')
                failed += 1
    except smtplib.SMTPAuthenticationError:
        print('Auth failed. Use a Gmail App Password (not your regular password).')
        print('Get one at: myaccount.google.com/apppasswords')
    finally:
        try: server.quit()
        except: pass

    print(f'\nDone: {sent} sent, {failed} failed.')
    if sent == len(TEST_EMAILS):
        print('Inbox ready — run the Refund Agent AUTO mode below.')

In [ ]:
async def main_refund_agent(mode: str = 'auto') -> None:
    """Orchestrator for the Refund Email Agent.
    mode='auto'        — run autonomous email processing
    mode='interactive' — start interactive chat loop
    """
    missing = [k for k in ['GOOGLE_OAUTH_CLIENT_ID', 'GOOGLE_OAUTH_CLIENT_SECRET']
               if not os.environ.get(k)]
    if missing:
        print('Cannot start — missing environment variables:')
        for k in missing:
            print(f'  {k}')
        _print_setup_guide()
        return

    print(f'Starting Refund Email Agent (mode={mode})...')
    async with MultiServerMCPClient(GMAIL_MCP_CONFIG) as mcp_client:
        agent = await build_gmail_agent(mcp_client)
        if mode == 'auto':
            await run_auto_refund_processing(agent)
        else:
            await run_interactive_gmail_chat(agent)


print('main_refund_agent() defined.')
print('Call: asyncio.run(main_refund_agent("auto")) or asyncio.run(main_refund_agent("interactive"))')

### 6.14 _print_setup_guide() — Developer Helper (Both Agents)

Prints a complete terminal guide covering **installation**, **Google Cloud OAuth setup**, **scope configuration**, **environment variable export**, and **example CLI verification commands**. Called only when environment variable validation fails in `main()`.

In [ ]:
def _print_setup_guide() -> None:
    """Print the full developer setup guide when required env vars are missing."""
    print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         AI Workspace Agent Suite — Setup Guide                          ║
╚══════════════════════════════════════════════════════════════════════════╝

Step 1 — Install uv (required for uvx workspace-mcp)
  Windows (PowerShell):
    powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
  macOS/Linux:
    curl -LsSf https://astral.sh/uv/install.sh | sh

Step 2 — Clone and install workspace-mcp + CLI
  git clone https://github.com/taylorwilsdon/google_workspace_mcp
  cd google_workspace_mcp
  uv tool install .          # installs workspace-cli globally
  pip install workspace-mcp

Step 3 — Install Python dependencies
  pip install langgraph langchain-openai langchain-mcp-adapters nest_asyncio

Step 4 — Google Cloud OAuth Setup
  a) console.cloud.google.com → New project
  b) Enable: Gmail API + Google Calendar API
  c) OAuth consent screen → External → add your email as test user
  d) Scopes: gmail.modify, gmail.send, calendar, calendar.events
  e) Create OAuth 2.0 credentials → Desktop App → copy Client ID + Secret

Step 5 — Set environment variables
  GOOGLE_OAUTH_CLIENT_ID      = <from Google Cloud Console>
  GOOGLE_OAUTH_CLIENT_SECRET  = <from Google Cloud Console>
  OAUTHLIB_INSECURE_TRANSPORT = 1   (local dev only)
  OPENAI_API_KEY              = sk-...  (required — from platform.openai.com)

  In this notebook: fill in the Credentials cell above and re-run.
  Persistent storage: add to your .env file (gitignored).

Step 6 — Verify CLI before running agents
  workspace-cli list                    # list all available tools
  workspace-cli call list_calendars     # verify Calendar OAuth works
  workspace-cli call list_gmail_labels  # verify Gmail OAuth works

Step 7 — First OAuth run
  The first time an agent runs, a browser window opens for Google consent.
  Approve once — tokens cached at ~/.workspace-mcp/ (Fernet-encrypted).
""")


print('_print_setup_guide() defined.')
print('Called automatically by main_refund_agent() and main_calendar_agent() when env vars are missing.')

### 7 — Data Flow: Refund Agent — "Process all refund emails"

Expected ReAct trace:
1. `agent_node` (turn 1) → model emits: `search_gmail_messages(query='refund OR return is:unread')`
2. `tool_node` → MCP client → workspace-mcp → Gmail API → returns email list
3. `agent_node` (turn 2) → model: "Found N emails. Reading first." → `get_gmail_message_content(message_id=...)`
4. `tool_node` → returns full email body, sender, `thread_id`
5. `agent_node` (turn 3) → model: "REFUND_REQUEST. Sending reply." → `send_gmail_message(to=..., thread_id=..., body=...)`
6. Repeats for remaining emails
7. `agent_node` (final turn) → model: no tool_calls → final summary report
8. `should_continue` → END

In [ ]:
# ── Run the Refund Email Agent in AUTO mode ───────────────────────────────────
# Requires: OPENAI_API_KEY, GOOGLE_OAUTH_CLIENT_ID, GOOGLE_OAUTH_CLIENT_SECRET set in .env
# First run will open a browser window for Google OAuth consent (one-time)
asyncio.run(main_refund_agent(mode='auto'))

In [ ]:
# ── Run the Refund Email Agent in INTERACTIVE mode ────────────────────────────
# Uncomment to start the interactive chat loop (type 'exit' to quit)
# asyncio.run(main_refund_agent(mode='interactive'))

---
## 8 — Project B: Calendar Agent

An interactive AI assistant that answers natural language questions about a Google Calendar, creates and modifies events, checks for free time slots, and sends RSVPs — using both **MCP tools** and lightweight **CLI bash tools** depending on query complexity.

**Dual tool strategy**:
- Simple read → `workspace-cli` bash tool (fast subprocess, minimal overhead)
- Create/Edit → Calendar MCP tool (full CRUD, rich JSON response)

### 6.4 _run_cli() — CLI Subprocess Runner (Calendar Agent)

A private synchronous helper that runs `workspace-cli <args>` as a subprocess. It captures stdout, attempts `json.loads()` on the output, and falls back to raw text if parsing fails.

Handles three error conditions: non-zero exit code, timeout (default 15s), and `FileNotFoundError` when `workspace-cli` is not installed.

In [ ]:
def _run_cli(args: list[str], timeout: int = 15) -> dict[str, Any]:
    """Run workspace-cli <args> as a subprocess and return parsed JSON output."""
    try:
        result = subprocess.run(
            ['workspace-cli'] + args,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode != 0:
            return {'error': result.stderr.strip() or f'Exit code {result.returncode}'}
        try:
            return json.loads(result.stdout)
        except json.JSONDecodeError:
            return {'raw': result.stdout.strip()}
    except subprocess.TimeoutExpired:
        return {'error': f'CLI call timed out after {timeout}s'}
    except FileNotFoundError:
        return {
            'error': 'workspace-cli not found.',
            'fix': 'Install with: git clone https://github.com/taylorwilsdon/google_workspace_mcp && uv tool install .'
        }


print('_run_cli() helper defined.')
print('  Handles: non-zero exit, timeout (15s), missing executable')

### 6.5 CLI @tool Functions — Calendar Agent

Five Python functions decorated with `@tool`. The decorator exposes each function's name, type signature, and docstring to the model as a callable tool schema.
All five delegate to `_run_cli()` — they only differ in arguments and docstring.

In [ ]:
@tool
def cli_today_events(calendar_id: str = 'primary') -> str:
    """List today's calendar events using the workspace-cli (fast read, no MCP roundtrip)."""
    now = datetime.now(timezone.utc)
    time_min = now.replace(hour=0, minute=0, second=0, microsecond=0).isoformat()
    time_max = now.replace(hour=23, minute=59, second=59, microsecond=0).isoformat()
    result = _run_cli([
        'call', 'list_calendar_events',
        '--calendarId', calendar_id,
        '--timeMin', time_min,
        '--timeMax', time_max,
        '--singleEvents', 'true',
        '--orderBy', 'startTime',
    ])
    return json.dumps(result, indent=2, default=str)


@tool
def cli_list_events(
    time_min: str,
    time_max: str = None,
    max_results: int = 10,
    calendar_id: str = 'primary',
) -> str:
    """List calendar events in a date range using the workspace-cli. time_min/max in ISO 8601 format."""
    if time_max is None:
        time_max = (datetime.now(timezone.utc) + timedelta(days=7)).isoformat()
    result = _run_cli([
        'call', 'list_calendar_events',
        '--calendarId', calendar_id,
        '--timeMin', time_min,
        '--timeMax', time_max,
        '--maxResults', str(max_results),
        '--singleEvents', 'true',
        '--orderBy', 'startTime',
    ])
    return json.dumps(result, indent=2, default=str)


@tool
def cli_list_calendars() -> str:
    """List all Google Calendars using the workspace-cli (fast, no MCP roundtrip)."""
    return json.dumps(_run_cli(['call', 'list_calendars']), indent=2, default=str)


@tool
def cli_get_event(event_id: str, calendar_id: str = 'primary') -> str:
    """Get full details for a specific calendar event using the workspace-cli."""
    result = _run_cli([
        'call', 'get_calendar_event',
        '--calendarId', calendar_id,
        '--eventId', event_id,
    ])
    return json.dumps(result, indent=2, default=str)


@tool
def cli_tool_list() -> str:
    """List all tools available from the workspace-cli server (for debugging and tool discovery)."""
    return json.dumps(_run_cli(['list']), indent=2, default=str)


CLI_TOOLS = [cli_today_events, cli_list_events, cli_list_calendars, cli_get_event, cli_tool_list]
print(f'All 5 CLI @tool functions defined:')
for t in CLI_TOOLS:
    print(f'  - {t.name}')

### WORKSPACE_MCP_CONFIG — Calendar Agent

Same structure as Gmail config, but with `--permissions calendar` to restrict scope to Calendar only.

In [ ]:
CALENDAR_MCP_CONFIG = {
    'workspace': {
        'command': 'uvx',
        'args': [
            'workspace-mcp',
            '--single-user',
            '--tool-tier', 'core',
            '--permissions', 'calendar',
        ],
        'transport': 'stdio',
        'env': {
            'GOOGLE_OAUTH_CLIENT_ID':     os.environ.get('GOOGLE_OAUTH_CLIENT_ID', ''),
            'GOOGLE_OAUTH_CLIENT_SECRET': os.environ.get('GOOGLE_OAUTH_CLIENT_SECRET', ''),
        },
    }
}

CALENDAR_TOOL_NAMES = [
    'list_calendar_events',
    'get_calendar_event',
    'list_calendars',
    'create_calendar_event',
    'update_calendar_event',
    'delete_calendar_event',
    'suggest_meeting_time',
    'respond_to_calendar_event',
]

print('Calendar MCP config ready.')
print(f'  Permissions : calendar')
print(f'  MCP tools   : {len(CALENDAR_TOOL_NAMES)}')
print(f'  CLI tools   : {len(CLI_TOOLS)}')
print(f'  Total tools : {len(CALENDAR_TOOL_NAMES) + len(CLI_TOOLS)}')

### 6.3 SYSTEM_PROMPT — Calendar Agent

Includes today's date, explains the **CLI vs MCP tool selection guide**, output formatting rules (ISO to human-readable), and confirmation requirements before destructive operations.

In [ ]:
today_str = datetime.now().strftime('%A, %B %d, %Y')

CALENDAR_SYSTEM_PROMPT = SystemMessage(content=f"""
You are an intelligent Google Calendar assistant. Today is {today_str}.

## Tool Selection Guide
Use CLI tools for fast, read-only queries (no MCP roundtrip overhead):
  cli_today_events    — "What's on today?"
  cli_list_events     — "Show me this week" / "Events in date range"
  cli_list_calendars  — "What calendars do I have?"
  cli_get_event       — "Get details for that meeting"
  cli_tool_list       — Debug / tool discovery

Use MCP tools for CRUD operations (full API access):
  create_calendar_event      — "Schedule a meeting"
  update_calendar_event      — "Move/reschedule an event"
  delete_calendar_event      — "Cancel an event"
  suggest_meeting_time       — "Find a free slot for all attendees"
  respond_to_calendar_event  — "Accept/decline an invitation"

## Output Formatting Rules
- Convert ISO timestamps to human-readable: "Monday, May 20 at 2:00 PM"
- For event lists: show time, title, location (if any), attendees (if any)
- For free slot searches: list all available options with specific times
- Use the user's local timezone if detectable, otherwise UTC

## Safety Rules
- ALWAYS confirm with the user before deleting or updating an event
- For recurring events: clarify scope (this event only / this and following / all events)
- When suggesting meeting times: check attendee availability before proposing
- Never delete without explicit confirmation even if the user said so previously
""")

print('Calendar system prompt defined.')
print(f'  Date context: {today_str}')
print('  Tool selection: CLI for reads, MCP for CRUD')
print('  Safety: confirm before delete/update')

### 6.6 build_calendar_agent() — Agent Factory (Calendar Agent)

Same pattern as the Gmail agent factory, but merges **MCP tools + CLI @tool functions** into a single `all_tools` list. `ToolNode` handles both tool surfaces transparently — The model does not know or care whether a tool is backed by MCP or a subprocess.

In [ ]:
async def build_calendar_agent(mcp_client: MultiServerMCPClient):
    """Build the Calendar Agent LangGraph, merging MCP tools + CLI bash tools."""
    all_mcp_tools = await mcp_client.get_tools()
    calendar_mcp_tools = [t for t in all_mcp_tools if t.name in CALENDAR_TOOL_NAMES]

    all_tools = calendar_mcp_tools + CLI_TOOLS  # MCP CRUD + CLI reads

    print(f'MCP tools loaded: {len(all_mcp_tools)} total, {len(calendar_mcp_tools)} Calendar MCP selected.')
    print(f'CLI tools: {len(CLI_TOOLS)}')
    print(f'Total tools bound to {OPENAI_MODEL}: {len(all_tools)}')
    print('  MCP:', [t.name for t in calendar_mcp_tools])
    print('  CLI:', [t.name for t in CLI_TOOLS])

    llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    llm_with_tools = llm.bind_tools(all_tools)

    def agent_node(state: AgentState) -> AgentState:
        """LLM reasoning node: prepend system prompt, call gpt-4o."""
        response = llm_with_tools.invoke([CALENDAR_SYSTEM_PROMPT] + list(state['messages']))
        return {'messages': [response]}

    def should_continue(state: AgentState) -> str:
        """Route to tools if the model emitted tool calls, else END."""
        last = state['messages'][-1]
        if hasattr(last, 'tool_calls') and last.tool_calls:
            return 'tools'
        return END

    tool_node = ToolNode(all_tools)

    workflow = StateGraph(AgentState)
    workflow.add_node('agent', agent_node)
    workflow.add_node('tools', tool_node)
    workflow.set_entry_point('agent')
    workflow.add_conditional_edges('agent', should_continue)
    workflow.add_edge('tools', 'agent')

    return workflow.compile()


print('build_calendar_agent() defined.')
print('Graph: START -> agent_node <-> tool_node -> END (ReAct loop)')
print('tool_node handles both MCP tools and CLI @tool functions transparently.')

### 6.11 run_demo() — Demo Mode (Calendar Agent)

Iterates over three pre-written demo queries matching the PDF example interactions:
1. `"What calendars do I have?"` — tests `cli_list_calendars`
2. `"What's on my calendar today?"` — tests `cli_today_events`
3. `"Show me my events for the next 7 days"` — tests `cli_list_events`

Demonstrates CLI and MCP tool selection without requiring user input.

In [ ]:
async def run_calendar_demo(agent) -> None:
    """Run 3 pre-written demo queries to verify OAuth + CLI + MCP are working."""
    # Testing Specification prompts (Testing_project_2.pdf §1.4–1.6)
    demo_queries = [
        "What's on my calendar?",
        'Schedule a team lunch for the coming Friday at noon for 1 hour.',
        'Find a free 30-minute slot for a call with john@example.com this week.',
    ]

    print('Calendar Agent — Demo Mode')
    print('=' * 70)

    for i, query in enumerate(demo_queries, 1):
        print(f'\nDemo {i}: "{query}"')
        print('-' * 50)
        result = await agent.ainvoke({'messages': [HumanMessage(content=query)]})
        response = next(
            (m.content for m in reversed(result['messages'])
             if isinstance(m, AIMessage) and not getattr(m, 'tool_calls', None)),
            '(no response)'
        )
        print(f'Agent: {response}')

    print('\nDemo complete.')


print('run_calendar_demo() defined.')
print('Demo queries: calendars | today events | next 7 days events')

### 6.12 run_interactive_chat() — Interactive Mode (Calendar Agent)

Runs a `while True` CLI input loop. Maintains a `history` list for multi-turn conversation. Calendar Agent additionally handles `"demo"` as a special command that runs `run_calendar_demo()` inline.

In [ ]:
async def run_interactive_calendar_chat(agent) -> None:
    """Interactive multi-turn chat with the Calendar Agent."""
    history: list[BaseMessage] = []
    print('Calendar Agent — Interactive Mode')
    print('Commands: type your query | "demo" to run demo | "exit" to quit')
    print('=' * 60)

    while True:
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            break

        if not user_input:
            continue
        if user_input.lower() in ('exit', 'quit'):
            print('Goodbye!')
            break
        if user_input.lower() == 'demo':
            await run_calendar_demo(agent)
            continue

        history.append(HumanMessage(content=user_input))
        result = await agent.ainvoke({'messages': history})
        history = list(result['messages'])

        response = next(
            (m.content for m in reversed(history)
             if isinstance(m, AIMessage) and not getattr(m, 'tool_calls', None)),
            '(no response)'
        )
        print(f'Agent: {response}')
        print()


print('run_interactive_calendar_chat() defined.')

### 6.13 main() — Orchestrator (Calendar Agent)

Same structure as the Gmail orchestrator. Routes to `run_calendar_demo()` (default) or `run_interactive_calendar_chat()` based on mode argument.

In [ ]:
async def main_calendar_agent(mode: str = 'demo') -> None:
    """Orchestrator for the Calendar Agent.
    mode='demo'        — run 3 pre-written demo queries
    mode='interactive' — start interactive chat loop
    """
    missing = [k for k in ['GOOGLE_OAUTH_CLIENT_ID', 'GOOGLE_OAUTH_CLIENT_SECRET']
               if not os.environ.get(k)]
    if missing:
        print('Cannot start — missing environment variables:')
        for k in missing:
            print(f'  {k}')
        _print_setup_guide()
        return

    print(f'Starting Calendar Agent (mode={mode})...')
    async with MultiServerMCPClient(CALENDAR_MCP_CONFIG) as mcp_client:
        agent = await build_calendar_agent(mcp_client)
        if mode == 'demo':
            await run_calendar_demo(agent)
        else:
            await run_interactive_calendar_chat(agent)


print('main_calendar_agent() defined.')
print('Call: asyncio.run(main_calendar_agent("demo")) or asyncio.run(main_calendar_agent("interactive"))')

### 7 — Data Flow: Calendar Agent — "What's on today?"

Expected ReAct trace (from PDF Section 7):
1. `agent_node` (turn 1) → model: "Simple read — use CLI tool for speed" → `cli_today_events(calendar_id='primary')`
2. `tool_node` → `_run_cli(['call', 'list_calendar_events', ...])` → subprocess → `workspace-cli` → Google Calendar API
3. Returns JSON with today's events
4. `agent_node` (turn 2) → model: no tool_calls → "You have 3 events today: 9:00 AM Standup..."
5. `should_continue` → END

---
### Step 3 — Populate Calendar with Test Events

**Testing Specification §1.2** requires these 10 events to exist before running the Calendar Agent demo.
Run this cell to create them via `workspace-cli` (uses the same OAuth session).

| Date | Events |
| --- | --- |
| Mon Jun 1 | Team Standup 09:00, Research Meeting 14:00 |
| Tue Jun 2 | Project Review 10:00, Student Advising 15:00 |
| Wed Jun 3 | Faculty Meeting 09:00, PhD Progress Review 14:00 |
| Thu Jun 4 | Industry Collaboration Meeting 11:00, Lab Weekly Meeting 15:00 |
| Fri Jun 5 | Grant Proposal Discussion 09:00, Research Seminar 15:00 |

In [ ]:
import subprocess, json, shutil

TIMEZONE = 'Asia/Taipei'

TEST_EVENTS = [
    ('Team Standup',                   '2026-06-01T09:00:00', '2026-06-01T10:00:00'),
    ('Research Meeting',               '2026-06-01T14:00:00', '2026-06-01T15:00:00'),
    ('Project Review',                 '2026-06-02T10:00:00', '2026-06-02T11:00:00'),
    ('Student Advising',               '2026-06-02T15:00:00', '2026-06-02T16:00:00'),
    ('Faculty Meeting',                '2026-06-03T09:00:00', '2026-06-03T10:30:00'),
    ('PhD Progress Review',            '2026-06-03T14:00:00', '2026-06-03T15:00:00'),
    ('Industry Collaboration Meeting', '2026-06-04T11:00:00', '2026-06-04T12:00:00'),
    ('Lab Weekly Meeting',             '2026-06-04T15:00:00', '2026-06-04T16:00:00'),
    ('Grant Proposal Discussion',      '2026-06-05T09:00:00', '2026-06-05T10:00:00'),
    ('Research Seminar',               '2026-06-05T15:00:00', '2026-06-05T16:00:00'),
]

if not shutil.which('workspace-cli'):
    print('workspace-cli not found — install it first')
else:
    created, failed = 0, 0
    print(f'Creating {len(TEST_EVENTS)} test calendar events via workspace-cli...\n')
    for idx, (summary, start, end) in enumerate(TEST_EVENTS):
        payload = json.dumps({
            'summary': summary,
            'start': {'dateTime': f'{start}+08:00', 'timeZone': TIMEZONE},
            'end':   {'dateTime': f'{end}+08:00',   'timeZone': TIMEZONE},
        })
        result = subprocess.run(
            ['workspace-cli', 'call', 'create_calendar_event',
             '--summary', summary,
             '--start', f'{start}+08:00',
             '--end',   f'{end}+08:00',
             '--timezone', TIMEZONE],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0:
            print(f'[{idx+1}/{len(TEST_EVENTS)}] OK  {summary}')
            created += 1
        else:
            # Try alternative format
            result2 = subprocess.run(
                ['workspace-cli', 'call', 'create_calendar_event',
                 f'summary={summary}',
                 f'start={start}+08:00',
                 f'end={end}+08:00'],
                capture_output=True, text=True, timeout=30
            )
            if result2.returncode == 0:
                print(f'[{idx+1}/{len(TEST_EVENTS)}] OK  {summary}')
                created += 1
            else:
                print(f'[{idx+1}/{len(TEST_EVENTS)}] FAIL {summary}: {(result.stderr or result2.stderr)[:80]}')
                failed += 1

    print(f'\nDone: {created} created, {failed} failed.')
    if failed > 0:
        print('\nIf workspace-cli format differs, run Project-2/createcalendarevents.py directly:')
        print('  python Project-2/createcalendarevents.py')
        print('  (requires: pip install google-auth google-auth-oauthlib google-api-python-client)')
        print('  (requires: token.json in current directory — copy from workspace-mcp token location)')

In [ ]:
# ── Run the Calendar Agent in DEMO mode (3 pre-written queries) ───────────────
# Requires: OPENAI_API_KEY, GOOGLE_OAUTH_CLIENT_ID, GOOGLE_OAUTH_CLIENT_SECRET set in .env
# First run will open a browser window for Google OAuth consent (one-time)
asyncio.run(main_calendar_agent(mode='demo'))

In [ ]:
# ── Run the Calendar Agent in INTERACTIVE mode ────────────────────────────────
# Uncomment to start the interactive chat loop (type 'demo' or 'exit')
# asyncio.run(main_calendar_agent(mode='interactive'))

---
## 8 — Security Design *(Informational — no code required)*

| Concern | How It Is Addressed |
| --- | --- |
| OAuth credentials | Stored in env vars, never hardcoded. Tokens cached encrypted at `~/.workspace-mcp/` using a Fernet key. |
| Permission scope | `--permissions gmail:send` or `calendar` — each agent loads only what it needs. No admin, no Drive writes. |
| Data locality | MCP server runs locally via stdio. Gmail/Calendar data never passes through a third-party server. |
| Auto-send guard | Refund agent system prompt instructs: prefer `create_gmail_draft` over `send_gmail_message` when uncertain. |
| CLI timeout | `_run_cli()` enforces a 15-second timeout on every subprocess call to prevent hanging. |
| Destructive ops | Calendar agent system prompt requires explicit confirmation before delete or update operations. |

---
## 11 — Key Concepts Reference *(Informational — no code required)*

| Concept | Definition | Where Used |
| --- | --- | --- |
| ReAct pattern | Reason → Act → Observe loop; agent alternates thinking and tool use | Both agents |
| LangGraph StateGraph | Directed graph of nodes (functions) connected by typed edges | `build_agent()` |
| MCP (Model Context Protocol) | Open standard JSON-RPC protocol for AI tool access | `WORKSPACE_MCP_CONFIG` |
| Tool binding | Attaching tool schemas to an LLM so it can emit structured function calls | `llm.bind_tools()` |
| stdio transport | MCP communication via stdin/stdout pipe — runs entirely locally | MCP config |
| TypedDict | Python type hint for dicts with fixed keys; used for state schema | `AgentState` |
| `add_messages` reducer | LangGraph helper that appends to the message list instead of replacing it | `AgentState` |
| Conditional edge | LangGraph routing that dynamically chooses the next node based on state | `should_continue()` |
| OAuth 2.0 | Delegated access standard; user grants scoped permissions without sharing password | Google Cloud setup |
| Fernet encryption | Symmetric encryption used by workspace-cli for local token caching | Token storage |
| Thread ID | Gmail identifier that groups messages in the same email conversation | `send_gmail_message` |
| `tool_tier core` | workspace-mcp flag that loads ~20 essential tools instead of all 100+ | MCP config |
| `@tool` decorator | LangChain decorator that converts a Python function into an LLM-callable tool | CLI bash tools |
| subprocess | Python module used to spawn and communicate with the workspace-cli process | `_run_cli()` |

---
## Summary — All PDF Sections Covered

| # | PDF Section | Status | Evidence |
| --- | --- | --- | --- |
| 1 | Project Overview | Markdown | Section 1 cell |
| 2 | Projects in the Suite (A + B) | Markdown | Section 2 cell |
| 3 | Technology Stack | Markdown | Section 3 cell |
| 4 | System Architecture | Code | `build_gmail_agent()`, `build_calendar_agent()` |
| 5.1 | Gmail MCP Tools | Code | `GMAIL_TOOL_NAMES` (7 tools) |
| 5.2 | Calendar MCP Tools | Code | `CALENDAR_TOOL_NAMES` (8 tools) |
| 5.3 | workspace-cli Bash Tools | Code | 5 `@tool` functions (`CLI_TOOLS`) |
| 6.1 | AgentState | Code | `AgentState` TypedDict with `add_messages` reducer |
| 6.2 | MCP Config | Code | `GMAIL_MCP_CONFIG`, `CALENDAR_MCP_CONFIG` |
| 6.3 | System Prompts | Code | `GMAIL_SYSTEM_PROMPT`, `CALENDAR_SYSTEM_PROMPT` |
| 6.4 | `_run_cli()` | Code | CLI subprocess runner with timeout + error handling |
| 6.5 | CLI `@tool` Functions | Code | `cli_today_events`, `cli_list_events`, `cli_list_calendars`, `cli_get_event`, `cli_tool_list` |
| 6.6 | `build_agent()` | Code | `build_gmail_agent()`, `build_calendar_agent()` |
| 6.7 | `agent_node` | Code | Defined inside `build_*_agent()` with closure over bound LLM |
| 6.8 | `should_continue` | Code | Defined inside `build_*_agent()`, routes `"tools"` or `END` |
| 6.9 | `tool_node` | Code | `ToolNode(all_tools)` — handles MCP + CLI transparently |
| 6.10 | `run_auto_refund_processing()` | Code | Fires one instruction, awaits full ReAct loop |
| 6.11 | `run_demo()` | Code | `run_calendar_demo()` — 3 pre-written queries |
| 6.12 | `run_interactive_chat()` | Code | `run_interactive_gmail_chat()`, `run_interactive_calendar_chat()` |
| 6.13 | `main()` | Code | `main_refund_agent()`, `main_calendar_agent()` |
| 6.14 | `_print_setup_guide()` | Code | Prints full install + OAuth + env guide when creds missing |
| 7 | Data Flow | Markdown + run cells | Section 7 & 8 data flow cells + live agent runs |
| 8 | Security Design | Markdown | Section 8 cell |
| 9 | Setup Instructions | Code | Install cell + Credentials cell |
| 10 | Example Interactions | Code | `run_auto_refund_processing()`, `run_calendar_demo()` run cells |
| 11 | Key Concepts Reference | Markdown | Section 11 cell |

---

## Credentials Needed to Run

To activate both agents, provide these in the **Credentials cell** above or in your `.env` file:

| Variable | Source | Required For |
| --- | --- | --- |
| `DGX_SPARK_URL` | Default: `http://140.118.122.123:8090/v1` (no key needed) | Both agents (nemotron-super-fp8) |
| `GOOGLE_OAUTH_CLIENT_ID` | Google Cloud Console → OAuth 2.0 Client | Both agents (Gmail + Calendar) |
| `GOOGLE_OAUTH_CLIENT_SECRET` | Google Cloud Console → OAuth 2.0 Client | Both agents (Gmail + Calendar) |
| `OAUTHLIB_INSECURE_TRANSPORT=1` | Set to `1` for local dev | Both agents (skip HTTPS check) |

Also required (installed separately):
```bash
git clone https://github.com/taylorwilsdon/google_workspace_mcp
cd google_workspace_mcp && uv tool install .
pip install workspace-mcp
```